# Run one baseline on Kaggle

Select a scenario and either the 10-client or 100-client frozen partition. The notebook prints a dry-run plan by default; set `EXECUTE = True` only after the paths and plan are correct.

In [ ]:
from pathlib import Path
import subprocess
import sys

SCENARIO = 'fedavg'
NUM_CLIENTS = 10  # 10 or 100
SEED = 42
EXECUTE = False

REPO_ROOT = Path('/kaggle/working/ids_fed')
DATA_DIRS = {
    10: Path('/kaggle/input/your-dataset/prepared-10'),
    100: Path('/kaggle/input/your-dataset/prepared-100'),
}
DATA_DIR = DATA_DIRS[NUM_CLIENTS]
MANIFEST = REPO_ROOT / f'configs/partitions/{NUM_CLIENTS}_clients_manifest.json'
OUTPUT_ROOT = Path('/kaggle/working/results/baselines')


In [ ]:
SCENARIOS = {
    'fedavg', 'fedprox', 'bdd_hfl', 'bdd_hfl_mu',
    'fap', 'fap_mu', 'fedpaq', 'dadaquant',
}
if SCENARIO not in SCENARIOS:
    raise ValueError(f'Unknown scenario: {SCENARIO}')
if NUM_CLIENTS not in (10, 100):
    raise ValueError('NUM_CLIENTS must be 10 or 100')
for path in (REPO_ROOT, DATA_DIR, MANIFEST):
    if not path.exists():
        raise FileNotFoundError(path)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)], check=True)


In [ ]:
command = [
    sys.executable, str(REPO_ROOT / 'scripts/run_paper34_campaign.py'),
    '--scenario', SCENARIO,
    '--num-clients', str(NUM_CLIENTS),
    '--seed', str(SEED),
    '--partitions-dir', str(DATA_DIR),
    '--partition-file', str(MANIFEST),
    '--output-root', str(OUTPUT_ROOT),
    '--device', 'auto',
    '--parallel-clients', '2',
]
if EXECUTE:
    command.append('--execute')
print(' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)
